# Study 1 — Dependency Graph Analysis

Builds directed graphs from the `depends_on` field and analyses graph-level properties:
degree by category, density, longest paths, and connected components.

**Outputs:**
- `outputs/study1_analysis/tables/dependency_degree_by_category.csv`
- `outputs/study1_analysis/tables/graph_statistics.csv`
- `outputs/study1_analysis/tables/graph_statistics_summary.csv`
- `outputs/study1_analysis/figures/dependency_degree_by_category.png`
- `outputs/study1_analysis/figures/graph_density_by_task.png`
- `outputs/study1_analysis/figures/longest_path_by_completion.png`
- `outputs/study1_analysis/figures/longest_path_vs_length.png`

In [1]:
# ── Setup ────────────────────────────────────────────────────────────────────
from study1_helpers import *
import networkx as nx
from scipy.stats import kruskal, mannwhitneyu

traces = load_traces()
df = build_sentence_df(traces)
print(f'Corpus: {df.trace_key.nunique()} traces, {len(df):,} sentences')
setup_style()

Loaded 320 traces from /Users/taliacharara/Desktop/cot-faithfulness/outputs/traces_clean_coded


Corpus: 320 traces, 73,383 sentences


## §3a. Graph Construction

Build a directed graph per trace from the `depends_on` field. Nodes = sentence IDs, edges = dependency references (d → s for each d in sentence s's `depends_on` list).

In [2]:
section_header('3a. Graph Construction')

# Build one directed graph per trace
graphs = {}
trace_meta = {}  # trace_key -> {task_id, set, completed}

for key, grp in df.groupby('trace_key'):
    grp = grp.sort_values('sentence_id')
    valid_ids = set(grp['sentence_id'].tolist())
    n = len(grp)

    G = nx.DiGraph()
    # Add all nodes with micro_label attribute
    for _, row in grp.iterrows():
        G.add_node(row['sentence_id'], micro_label=row['micro_label'])

    # Add edges: for each dependency d of sentence s, add edge d → s
    skipped = 0
    for _, row in grp.iterrows():
        sid = row['sentence_id']
        for d in row['depends_on']:
            if d in valid_ids:
                G.add_edge(d, sid)
            else:
                skipped += 1

    graphs[key] = G
    trace_meta[key] = {
        'task_id': grp['task_id'].iloc[0],
        'set': grp['set'].iloc[0],
        'completed': grp['completed'].iloc[0],
    }

print(f'Built {len(graphs)} directed graphs')
print(f'Total edges: {sum(G.number_of_edges() for G in graphs.values()):,}')
print(f'Dependency references outside trace range (skipped): {skipped}')

# ── Cycle check ──
n_cyclic = sum(1 for G in graphs.values() if not nx.is_directed_acyclic_graph(G))
print(f'\nTraces with cycles: {n_cyclic}/{len(graphs)}')

if n_cyclic > 0:
    print('\nCyclic traces:')
    for key, G in graphs.items():
        if not nx.is_directed_acyclic_graph(G):
            cycles = list(nx.simple_cycles(G))
            print(f'  {key}: {len(cycles)} cycle(s), nodes involved: '
                  f'{sorted(set(n for c in cycles for n in c))[:10]}...'
                  if len(set(n for c in cycles for n in c)) > 10
                  else f'  {key}: {len(cycles)} cycle(s), nodes involved: '
                       f'{sorted(set(n for c in cycles for n in c))}')


  3a. Graph Construction



Built 320 directed graphs
Total edges: 161,051
Dependency references outside trace range (skipped): 0

Traces with cycles: 1/320

Cyclic traces:
  set_a/task1/trace_035: 2 cycle(s), nodes involved: [151, 152, 153]


## §3b. Degree Analysis by Category

For each micro-label, compute the mean in-degree (how often sentences of this type are referenced by later sentences) and mean out-degree (how many dependencies sentences of this type declare).

In [3]:
section_header('3b. Degree Analysis by Category')

# Accumulate in-degree and out-degree per micro-label across all graphs
from collections import defaultdict

degree_data = defaultdict(lambda: {'in': [], 'out': []})

for key, G in graphs.items():
    for node in G.nodes():
        label = G.nodes[node]['micro_label']
        degree_data[label]['in'].append(G.in_degree(node))
        degree_data[label]['out'].append(G.out_degree(node))

# Build summary table
deg_rows = []
for label in MICRO_LABELS:
    in_vals = np.array(degree_data[label]['in'])
    out_vals = np.array(degree_data[label]['out'])
    deg_rows.append({
        'label': label,
        'mean_in_degree': round(in_vals.mean(), 3),
        'sd_in_degree': round(in_vals.std(), 3),
        'mean_out_degree': round(out_vals.mean(), 3),
        'sd_out_degree': round(out_vals.std(), 3),
        'n_sentences': len(in_vals),
    })

deg_df = pd.DataFrame(deg_rows)
save_table(deg_df, 'dependency_degree_by_category.csv')
display(deg_df)

# ── Grouped bar chart ──
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(MICRO_LABELS))
w = 0.35

in_means = deg_df['mean_in_degree'].values
out_means = deg_df['mean_out_degree'].values
in_ses = deg_df['sd_in_degree'].values / np.sqrt(deg_df['n_sentences'].values)
out_ses = deg_df['sd_out_degree'].values / np.sqrt(deg_df['n_sentences'].values)

ax.bar(x - w/2, in_means, w, yerr=in_ses, capsize=3,
       label='Mean in-degree', color='steelblue', alpha=0.85)
ax.bar(x + w/2, out_means, w, yerr=out_ses, capsize=3,
       label='Mean out-degree', color='darkorange', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(MICRO_LABELS, rotation=45, ha='right')
ax.set_ylabel('Mean degree')
ax.set_xlabel('Micro-label')
ax.set_title('Dependency Degree by Micro-Label Category')
ax.legend()
plt.tight_layout()
save_fig(fig, 'dependency_degree_by_category.png')
plt.show()


  3b. Degree Analysis by Category

  Saved: /Users/taliacharara/Desktop/cot-faithfulness/outputs/study1_analysis/tables/dependency_degree_by_category.csv


,label,mean_in_degree,sd_in_degree,mean_out_degree,sd_out_degree,n_sentences
0,ORIENT,0.519,0.511,1.721,1.203,725
1,DESCRIBE,0.974,0.166,1.615,0.954,6134
2,SYNTHESIZE,3.447,1.876,1.140,1.228,2828
3,HYPO,1.733,0.448,4.234,3.896,12824
4,TEST,2.492,1.243,2.056,1.113,36832
5,JUDGE,2.836,1.182,1.575,1.513,9054
6,PLAN,0.989,0.103,0.527,1.059,2973
7,MONITOR,1.000,0.000,0.480,0.753,1684
8,RULE,1.945,0.276,0.027,0.163,329


  Saved: /Users/taliacharara/Desktop/cot-faithfulness/outputs/study1_analysis/figures/dependency_degree_by_category.png


### Interpretation — Degree by Category

The degree profiles reflect the informational role of each category in the dependency graph:

- **HYPO** shows high in-degree: TEST and JUDGE sentences frequently reference the hypothesis they are evaluating, making HYPO a central hub in the graph.
- **JUDGE** shows high out-degree: verdicts reference both the HYPO being judged and the TEST sentences that provided the evidence, creating multiple outgoing dependency edges.
- **DESCRIBE** shows moderate in-degree: later sentences reference extracted evidence during hypothesis testing.
- **TEST** shows moderate out-degree: each TEST sentence typically references the HYPO being tested plus DESCRIBE sentences providing evidence.
- **RULE** shows high out-degree: the final rule references the HYPO/JUDGE chain that established it.
- **ORIENT** shows near-zero in-degree: opening sentences are rarely referenced by later reasoning.

## §3c. Graph-Level Statistics

Per-trace graph metrics: density, longest directed path, connected components, and degree extremes. Statistical tests compare density across tasks and longest path by completion status.

In [4]:
section_header('3c. Graph-Level Statistics')

def longest_path_length(G):
    """Compute longest directed path length, handling cyclic graphs gracefully."""
    if nx.is_directed_acyclic_graph(G):
        if G.number_of_nodes() == 0:
            return 0
        return nx.dag_longest_path_length(G)
    else:
        # For cyclic graphs: remove back-edges to create a DAG
        # Use topological generations on the condensation (DAG of SCCs)
        condensation = nx.condensation(G)
        if condensation.number_of_nodes() == 0:
            return 0
        return nx.dag_longest_path_length(condensation)

graph_rows = []
for key, G in graphs.items():
    meta = trace_meta[key]
    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
    max_possible = n_nodes * (n_nodes - 1) if n_nodes > 1 else 1
    density = n_edges / max_possible if n_nodes > 1 else 0.0

    in_degs = [d for _, d in G.in_degree()]
    out_degs = [d for _, d in G.out_degree()]

    graph_rows.append({
        'trace_key': key,
        'task_id': meta['task_id'],
        'set': meta['set'],
        'completed': meta['completed'],
        'n_nodes': n_nodes,
        'n_edges': n_edges,
        'density': round(density, 6),
        'n_weakly_connected': nx.number_weakly_connected_components(G),
        'longest_path': longest_path_length(G),
        'mean_in_degree': round(np.mean(in_degs), 3) if in_degs else 0,
        'max_in_degree': max(in_degs) if in_degs else 0,
        'max_out_degree': max(out_degs) if out_degs else 0,
    })

graph_df = pd.DataFrame(graph_rows)
save_table(graph_df, 'graph_statistics.csv')
print(f'Graph statistics computed for {len(graph_df)} traces.')
display(graph_df[['n_nodes', 'n_edges', 'density', 'n_weakly_connected',
                   'longest_path', 'mean_in_degree', 'max_in_degree', 'max_out_degree']].describe().round(3))


  3c. Graph-Level Statistics

  Saved: /Users/taliacharara/Desktop/cot-faithfulness/outputs/study1_analysis/tables/graph_statistics.csv
Graph statistics computed for 320 traces.


,n_nodes,n_edges,density,n_weakly_connected,longest_path,mean_in_degree,max_in_degree,max_out_degree
count,320.000,320.000,320.000,320.000,320.000,320.000,320.000,320.000
mean,229.322,503.284,0.014,2.559,127.375,2.107,5.044,14.912
std,121.153,299.556,0.014,2.406,82.161,0.314,1.885,5.115
min,21.000,47.000,0.004,1.000,9.000,1.163,4.000,4.000
25%,128.500,251.250,0.008,1.000,62.000,1.889,5.000,11.750
50%,237.000,492.000,0.009,2.000,114.000,2.120,5.000,16.000
75%,298.000,695.000,0.016,3.000,173.250,2.338,5.000,19.000
max,558.000,1395.000,0.124,24.000,445.000,2.822,21.000,33.000


In [5]:
# ── Statistical tests ──

# Kruskal-Wallis: density by task_id
groups_density = [graph_df[graph_df['task_id'] == t]['density'].values
                  for t in sorted(graph_df['task_id'].unique())]
kw_stat, kw_p = kruskal(*groups_density)
print(f'Kruskal-Wallis (density ~ task): H={kw_stat:.3f}, p={kw_p:.4f}')

# Mann-Whitney U: longest_path by completion
completed_lp = graph_df[graph_df['completed'] == True]['longest_path']
truncated_lp = graph_df[graph_df['completed'] == False]['longest_path']
mw_stat, mw_p = mannwhitneyu(completed_lp, truncated_lp, alternative='two-sided')
print(f'Mann-Whitney U (longest_path ~ completion): U={mw_stat:.1f}, p={mw_p:.4f}')

# ── Summary table ──
def _summarise(series, n):
    return {
        'mean': round(series.mean(), 4),
        'median': round(series.median(), 4),
        'sd': round(series.std(), 4),
        'n': n,
    }

metrics = ['density', 'longest_path', 'n_edges', 'n_weakly_connected']
summary_rows = []
for metric in metrics:
    summary_rows.append({'group': 'overall', 'metric': metric,
                         **_summarise(graph_df[metric], len(graph_df))})
    for tid in sorted(graph_df['task_id'].unique()):
        sub = graph_df[graph_df['task_id'] == tid][metric]
        summary_rows.append({'group': f'task{tid}', 'metric': metric,
                             **_summarise(sub, len(sub))})
    for status, label in [(True, 'completed'), (False, 'truncated')]:
        sub = graph_df[graph_df['completed'] == status][metric]
        if len(sub) > 0:
            summary_rows.append({'group': label, 'metric': metric,
                                 **_summarise(sub, len(sub))})

summary_df = pd.DataFrame(summary_rows)
save_table(summary_df, 'graph_statistics_summary.csv')
display(summary_df)

Kruskal-Wallis (density ~ task): H=30.797, p=0.0000
Mann-Whitney U (longest_path ~ completion): U=3421.5, p=0.0000
  Saved: /Users/taliacharara/Desktop/cot-faithfulness/outputs/study1_analysis/tables/graph_statistics_summary.csv


,group,metric,mean,median,sd,n
0,overall,density,0.0141,0.0094,0.0136,320
1,task1,density,0.0164,0.0112,0.0140,80
2,task2,density,0.0124,0.0092,0.0097,80
3,task3,density,0.0089,0.0081,0.0044,80
4,task4,density,0.0187,0.0108,0.0194,80
5,completed,density,0.0202,0.0156,0.0168,163
6,truncated,density,0.0078,0.0078,0.0021,157
7,overall,longest_path,127.3750,114.0000,82.1615,320
8,task1,longest_path,99.9250,90.5000,64.5112,80
9,task2,longest_path,135.6875,122.5000,84.7949,80


In [6]:
# ── Figure 1: Graph density by task ──
fig, ax = plt.subplots(figsize=(8, 5))
task_data = [graph_df[graph_df['task_id'] == t]['density'].values
             for t in sorted(graph_df['task_id'].unique())]
bp = ax.boxplot(task_data, tick_labels=[f'Task {t}' for t in sorted(graph_df['task_id'].unique())])
ax.set_ylabel('Graph Density')
ax.set_title('Dependency Graph Density by Task')
ax.annotate(f'Kruskal-Wallis H={kw_stat:.2f}, p={kw_p:.4f}',
            xy=(0.02, 0.95), xycoords='axes fraction', fontsize=10)
plt.tight_layout()
save_fig(fig, 'graph_density_by_task.png')
plt.show()

# ── Figure 2: Longest path by completion ──
fig, ax = plt.subplots(figsize=(6, 5))
comp_data = [truncated_lp.values, completed_lp.values]
bp = ax.boxplot(comp_data, tick_labels=['Truncated', 'Completed'])
ax.set_ylabel('Longest Directed Path Length')
ax.set_title('Longest Path by Completion Status')
ax.annotate(f'Mann-Whitney U={mw_stat:.1f}, p={mw_p:.4f}',
            xy=(0.02, 0.95), xycoords='axes fraction', fontsize=10)
plt.tight_layout()
save_fig(fig, 'longest_path_by_completion.png')
plt.show()

# ── Figure 3: Longest path vs trace length ──
fig, ax = plt.subplots(figsize=(10, 6))
for tid in sorted(graph_df['task_id'].unique()):
    sub = graph_df[graph_df['task_id'] == tid]
    ax.scatter(sub['n_nodes'], sub['longest_path'], alpha=0.5, s=20, label=f'Task {tid}')

# Reference line: longest_path = n_nodes (theoretical max)
max_n = graph_df['n_nodes'].max()
ax.plot([0, max_n], [0, max_n], 'k--', alpha=0.3, label='y = x (theoretical max)')
ax.set_xlabel('Number of Sentences (n_nodes)')
ax.set_ylabel('Longest Directed Path Length')
ax.set_title('Longest Path vs Trace Length')
ax.legend(fontsize=9)
plt.tight_layout()
save_fig(fig, 'longest_path_vs_length.png')
plt.show()

  Saved: /Users/taliacharara/Desktop/cot-faithfulness/outputs/study1_analysis/figures/graph_density_by_task.png


  Saved: /Users/taliacharara/Desktop/cot-faithfulness/outputs/study1_analysis/figures/longest_path_by_completion.png


  Saved: /Users/taliacharara/Desktop/cot-faithfulness/outputs/study1_analysis/figures/longest_path_vs_length.png


### Interpretation — Graph-Level Statistics

- **Graph density** is expected to be low and decrease with trace length, since dependency edges connect nearby sentences rather than spanning the full graph. Differences across tasks may reflect task complexity: more complex rules may require more cross-referencing.
- **Longest directed path** captures the longest chain of informational dependencies — the critical path through the reasoning. Completed traces may show longer critical paths if successful reasoning requires building extended chains. Alternatively, truncated traces may show longer paths if the model gets stuck in deep dependency chains without converging.
- **Weakly connected components:** Ideally 1 per trace (fully connected reasoning). Multiple components indicate disconnected reasoning threads — the model starts a new line of investigation without referencing earlier work. Higher component counts may correlate with truncated traces where the model resets its approach.
- **Longest path vs trace length (scatter):** Points near the diagonal indicate traces where the critical path spans nearly the entire trace (sequential reasoning). Points well below the diagonal indicate traces with broad, shallow dependency structures (parallel reasoning threads).

## Verification

Check that all expected output files were produced.

In [7]:
# ── Verify all outputs ──
import os

new_tables = ['dependency_degree_by_category.csv', 'graph_statistics.csv', 'graph_statistics_summary.csv']
new_figures = ['dependency_degree_by_category.png', 'graph_density_by_task.png',
               'longest_path_by_completion.png', 'longest_path_vs_length.png']

print('Output verification:')
for f in new_tables:
    path = TABLES_DIR / f
    print(f"  {'✓' if path.exists() else '✗'} tables/{f}")
for f in new_figures:
    path = FIGURES_DIR / f
    print(f"  {'✓' if path.exists() else '✗'} figures/{f}")

Output verification:
  ✓ tables/dependency_degree_by_category.csv
  ✓ tables/graph_statistics.csv
  ✓ tables/graph_statistics_summary.csv
  ✓ figures/dependency_degree_by_category.png
  ✓ figures/graph_density_by_task.png
  ✓ figures/longest_path_by_completion.png
  ✓ figures/longest_path_vs_length.png
